# Activity 2: Can multimodal AI predict symptom trajectories?

**NINR Artificial Intelligence Summer Research Intensive**

[Open in Google Colab](https://colab.research.google.com/github/sonya-j/ninr-ai-bootcamp/blob/main/notebooks/02_multimodal_symptom_trajectories.ipynb)

> **Research question:** Can multimodal AI predict tomorrow's symptom severity using patient-reported, wearable, EHR, and contextual data?

You do not need prior machine-learning experience. Run each cell from top to bottom, then use the checkboxes to decide what information the model can see.

## What you will learn

By the end of this activity, you will be able to:

- explain what **multimodal** data means;
- recognize a **trajectory** as repeated observations over time;
- define a prediction time and a next-day outcome;
- compare a simple baseline with a multimodal model;
- turn data modalities on and off and interpret what changes;
- identify missingness, leakage, representation, and access limitations.

## Before we begin: what data are these?

The linked real-world research destination is the **NIH All of Us Research Program**, which includes surveys, longitudinal EHR data, and Fitbit activity, sleep, and heart-rate data. Individual-level All of Us records must remain inside its secure Researcher Workbench, so they are **not** included in this public repository.

This notebook therefore uses a **fully synthetic teaching cohort**. It contains no real people and is not an export or statistical reconstruction of All of Us data. Every variable and generation step is documented. The synthetic cohort lets everyone complete the same lab; an authorized researcher can later adapt the workflow inside All of Us.

This activity predicts a score for learning purposes. It does **not** create a clinically valid symptom measure or a tool for patient care.

## 1. Set up Python

This cell imports the tools we need. Running an import is like taking a tool out of a toolbox.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

print("Setup complete.")

## 2. Load the teaching cohort

Each row represents **one participant on one day**. The outcome is that participant's symptom score on the following day.

In [ ]:
from pathlib import Path

local_path = Path("../data/processed/multimodal_symptom_trajectories/participant_day.csv")
github_url = (
    "https://raw.githubusercontent.com/sonya-j/ninr-ai-bootcamp/main/"
    "data/processed/multimodal_symptom_trajectories/participant_day.csv"
)
data = pd.read_csv(local_path if local_path.exists() else github_url)

print(f"Rows: {len(data):,}")
print(f"Participants: {data['participant_id'].nunique():,}")
print(f"Variables: {data.shape[1]}")
display(data.head())

## 3. Meet the four modalities

A **modality** is a type or source of information. The same participant-day can contain information from several sources.

In [ ]:
modalities = {
    "Patient-reported": ["pain_today", "fatigue_today", "mood_today", "sleep_quality_today", "symptom_score_today"],
    "Wearable": ["steps", "active_minutes", "resting_heart_rate", "sleep_minutes"],
    "EHR": ["chronic_condition_count", "medication_count", "recent_encounter", "medication_change", "care_message"],
    "Context": ["age_group", "weekend", "temperature_c", "air_quality_index", "social_support", "day_index"],
}

pd.DataFrame([
    {"Modality": name, "Example variables": ", ".join(columns)}
    for name, columns in modalities.items()
])

### Your turn

Before running the next cell, make a prediction: which modality will be most useful for predicting tomorrow's symptom score? Why?

## 4. Explore one person's trajectory

Choose a synthetic participant. The upper plot shows symptom severity over time. The lower plot shows two wearable measures.

In [ ]:
participant_picker = widgets.Dropdown(
    options=sorted(data['participant_id'].unique()),
    value='P0001',
    description='Participant:'
)
trajectory_output = widgets.Output()

def plot_trajectory(change=None):
    with trajectory_output:
        clear_output(wait=True)
        person = data[data['participant_id'] == participant_picker.value].sort_values('day_index')
        fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
        axes[0].plot(person['day_index'], person['symptom_score_today'], marker='o', linewidth=2)
        axes[0].set_ylabel('Symptom score (0-10)')
        axes[0].set_title(f"Daily trajectory for {participant_picker.value}")
        axes[0].grid(alpha=.25)
        axes[1].plot(person['day_index'], person['steps'], label='Steps')
        axes[1].set_ylabel('Steps/day')
        axes[1].set_xlabel('Study day')
        axes[1].grid(alpha=.25)
        second_axis = axes[1].twinx()
        second_axis.plot(person['day_index'], person['sleep_minutes'], color='tab:orange', label='Sleep')
        second_axis.set_ylabel('Sleep minutes')
        lines = axes[1].lines + second_axis.lines
        axes[1].legend(lines, [line.get_label() for line in lines], loc='upper right')
        plt.tight_layout()
        plt.show()

participant_picker.observe(plot_trajectory, names='value')
display(participant_picker, trajectory_output)
plot_trajectory()

### Notice the missing values

Wearables can be removed, uncharged, or unsynced. People may skip a daily survey. Missingness is part of the scientific problem—not just a coding inconvenience.

In [ ]:
missing = (data.isna().mean() * 100).sort_values(ascending=False)
display(missing[missing > 0].rename('Missing percent').round(1).to_frame())

## 5. Define the prediction task

At the end of day **t**, the model receives the selected information available through that day. It predicts `symptom_score_tomorrow`, measured on day **t + 1**.

We hold out entire participants for testing. This prevents the same person's records from appearing in both training and test data.

Our evaluation measure is **mean absolute error (MAE)**. An MAE of 1.0 means predictions are off by one symptom point on average. Lower is better.

In [ ]:
rng = np.random.default_rng(42)
participants = data['participant_id'].unique().copy()
rng.shuffle(participants)
cut = int(len(participants) * 0.75)
train_people = set(participants[:cut])
test_people = set(participants[cut:])
train = data[data['participant_id'].isin(train_people)].copy()
test = data[data['participant_id'].isin(test_people)].copy()
outcome = 'symptom_score_tomorrow'

baseline_prediction = test['symptom_score_today'].fillna(train['symptom_score_today'].median())
baseline_mae = mean_absolute_error(test[outcome], baseline_prediction)
print(f"Training participants: {len(train_people)}")
print(f"Test participants: {len(test_people)}")
print(f"Simple baseline MAE: {baseline_mae:.2f} symptom points")

## 6. Build your multimodal model

Check the modalities the model may use, choose a model, and select **Run model**. Start with all four modalities. Then turn one modality off and compare the MAE.

- **Ridge regression** is a stable linear model and easier to explain.
- **Random forest** can learn nonlinear patterns and interactions, but is harder to interpret.

In [ ]:
patient_box = widgets.Checkbox(value=True, description='Patient-reported')
wearable_box = widgets.Checkbox(value=True, description='Wearable')
ehr_box = widgets.Checkbox(value=True, description='EHR')
context_box = widgets.Checkbox(value=True, description='Context')
model_picker = widgets.Dropdown(
    options=['Ridge regression', 'Random forest'],
    value='Ridge regression',
    description='Model:'
)
run_button = widgets.Button(description='Run model', button_style='primary')
model_output = widgets.Output()
last_results = {}

def run_model(_):
    chosen = []
    selected_names = []
    for checked, name in [
        (patient_box.value, 'Patient-reported'),
        (wearable_box.value, 'Wearable'),
        (ehr_box.value, 'EHR'),
        (context_box.value, 'Context'),
    ]:
        if checked:
            chosen.extend(modalities[name])
            selected_names.append(name)

    with model_output:
        clear_output(wait=True)
        if not chosen:
            display(Markdown('**Choose at least one modality.**'))
            return

        categorical = [column for column in chosen if data[column].dtype == 'object']
        numeric = [column for column in chosen if column not in categorical]
        preprocessing = ColumnTransformer([
            ('numeric', Pipeline([
                ('fill_missing', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]), numeric),
            ('categorical', Pipeline([
                ('fill_missing', SimpleImputer(strategy='most_frequent')),
                ('one_hot', OneHotEncoder(handle_unknown='ignore')),
            ]), categorical),
        ])
        estimator = (
            Ridge(alpha=1.0)
            if model_picker.value == 'Ridge regression'
            else RandomForestRegressor(n_estimators=120, min_samples_leaf=5, random_state=42, n_jobs=-1)
        )
        model = Pipeline([('prepare', preprocessing), ('model', estimator)])
        model.fit(train[chosen], train[outcome])
        predictions = model.predict(test[chosen])
        mae = mean_absolute_error(test[outcome], predictions)
        last_results.clear()
        last_results.update({
            'model': model, 'features': chosen, 'predictions': predictions,
            'mae': mae, 'modalities': selected_names,
        })

        direction = 'better' if mae < baseline_mae else 'worse'
        display(Markdown(
            f"### Result: MAE = {mae:.2f} symptom points\n"
            f"Selected modalities: **{', '.join(selected_names)}**. "
            f"This is **{abs(mae - baseline_mae):.2f} points {direction}** than the simple baseline."
        ))
        sample = np.arange(min(90, len(predictions)))
        plt.figure(figsize=(10, 4))
        plt.plot(sample, test[outcome].iloc[sample], label='Observed tomorrow', linewidth=2)
        plt.plot(sample, predictions[sample], label='Predicted tomorrow', linewidth=2)
        plt.xlabel('First test participant-days')
        plt.ylabel('Symptom score (0-10)')
        plt.title('Observed and predicted next-day symptoms')
        plt.legend()
        plt.grid(alpha=.25)
        plt.show()

run_button.on_click(run_model)
display(
    widgets.HTML('<b>Data available to the model</b>'),
    widgets.HBox([patient_box, wearable_box, ehr_box, context_box]),
    model_picker, run_button, model_output
)

### Experiments to try

1. Run all four modalities with ridge regression.
2. Turn off patient-reported data. What changes?
3. Use only wearable data.
4. Add EHR and context to wearables.
5. Compare ridge regression with random forest.

Record the MAE after each run. Does adding a modality always improve performance?

## 7. Check performance across age groups

Run a model above first. Then run this cell. A single overall MAE can hide subgroup differences. The purpose is to ask a fairness question—not to conclude that age causes model error.

In [ ]:
if not last_results:
    print('Run the interactive model first.')
else:
    evaluation = test[['age_group', outcome]].copy()
    evaluation['prediction'] = last_results['predictions']
    evaluation['absolute_error'] = (evaluation[outcome] - evaluation['prediction']).abs()
    subgroup_mae = evaluation.groupby('age_group')['absolute_error'].agg(['mean', 'count']).round(2)
    display(subgroup_mae.rename(columns={'mean': 'MAE', 'count': 'participant-days'}))
    subgroup_mae['MAE'].plot(kind='bar', figsize=(7, 4), ylabel='MAE (symptom points)', title='Error by synthetic age group')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=.25)
    plt.show()

## 8. Interpret carefully

Discuss with a partner or team:

1. **Value:** Which modality added useful information in your experiments?
2. **Burden:** Which data would be most burdensome for patients or nurses to collect?
3. **Missingness:** Who might be excluded when a charged, synced wearable is required?
4. **Timing:** Were all predictors genuinely available by the end of the current day?
5. **Leakage:** Why must `symptom_score_tomorrow` never appear among the predictors?
6. **Actionability:** What nursing action could follow a prediction—and what evidence would be needed before acting?
7. **Synthetic limitation:** Which conclusions cannot be made because these records are simulated?

## Build from here

Choose one extension:

- predict the average symptom score over the next 3 days instead of tomorrow;
- compare participant-level and row-level train/test splitting;
- add a missingness indicator for wearable non-wear;
- plot errors over time to look for drift;
- define a clinically meaningful threshold and turn the task into classification;
- adapt the modality mapping to authorized All of Us tables inside the Researcher Workbench.

> **Takeaway:** Multimodal AI can combine complementary signals, but more data do not automatically create a useful, fair, or clinically actionable model. The research question, timing, missingness, validation design, and intended nursing action remain central.